### «Умный помощник» для оформления командировок

#### Установка зависимостей

In [1]:
%pip install -q langchain langgraph langchain-community chromadb sentence-transformers torch pandas openai python-dotenv langchain-openai

Note: you may need to restart the kernel to use updated packages.


#### Конфигурация

In [1]:
import os
from typing import Dict, List, Any
import pandas as pd
from io import StringIO
import traceback
from dotenv import load_dotenv

# Загружаем .env
load_dotenv()

# === Чтение конфигурации ===
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openrouter").lower()

# OpenRouter
OPENROUTER_API_KEY = os.getenv("API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-20b:free")

# Локальная (Ollama / LM Studio)
# LOCAL_MODEL_NAME = os.getenv("LOCAL_MODEL_NAME", "llama3.2")
# LOCAL_BASE_URL = os.getenv("LOCAL_BASE_URL", "http://localhost:11434")
# LOCAL_API_KEY = os.getenv("LOCAL_API_KEY", "ollama")

# Эмбеддинги
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

# Логирование
LOG_LEVEL = os.getenv("LOG_LEVEL", "info").upper()

# === Инициализация LLM ===
llm = None

if LLM_PROVIDER == "openrouter" and OPENROUTER_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        api_key=OPENROUTER_API_KEY, # type: ignore
        base_url=OPENROUTER_BASE_URL,
        model=OPENROUTER_MODEL,
        temperature=0.7
    )
    print(f"✅ LLM инициализирован: OpenRouter / {OPENROUTER_MODEL}")
    # Быстрый пинг
    try:
        response = llm.invoke("Ответь OK")
        print(f"✅ Пинг успешен: {response.content[:50]}")
    except Exception as e:
        print(f"❌ Пинг провален: {e}")
        
# elif LLM_PROVIDER == "local":
#     from langchain_openai import ChatOpenAI
#     llm = ChatOpenAI(
#         api_key=LOCAL_API_KEY,
#         base_url=LOCAL_BASE_URL,
#         model=LOCAL_MODEL_NAME,
#         temperature=0.7
#     )
#     print(f"✅ LLM инициализирован: локальная модель {LOCAL_MODEL_NAME} ({LOCAL_BASE_URL})")
#     # Быстрый пинг
#     try:
#         response = llm.invoke("Ответь OK")
#         print(f"✅ Пинг успешен: {response.content[:50]}")
#     except Exception as e:
#         print(f"❌ Пинг провален: {e}")
        
else:
    print("⚠️ LLM не сконфигурирован (нет API ключа или выбран неподдерживаемый провайдер).")
    print("   Будет использована заглушка DummyLLM, которая не генерирует осмысленные ответы.")
    class DummyLLM:
        def invoke(self, prompt, **kwargs):
            return f"[DummyLLM] Нет реального LLM. Промпт: {prompt[:200]}..."
    llm = DummyLLM()

def print_config():
    print("\n" + "="*50)
    print("ТЕКУЩАЯ КОНФИГУРАЦИЯ")
    print("="*50)
    print(f"LLM Provider: {LLM_PROVIDER.upper()}")
    if LLM_PROVIDER == "openrouter":
        print(f"Model: {OPENROUTER_MODEL}")
        print(f"API Key: {'***' if OPENROUTER_API_KEY else 'НЕТ'}")
    # elif LLM_PROVIDER == "local":
    #     print(f"Model: {LOCAL_MODEL_NAME}")
    #     print(f"Base URL: {LOCAL_BASE_URL}")
    print(f"Embedding: {EMBEDDING_MODEL}")
    print("="*50 + "\n")

print_config()

c:\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM инициализирован: OpenRouter / openai/gpt-oss-20b:free
✅ Пинг успешен: OK

ТЕКУЩАЯ КОНФИГУРАЦИЯ
LLM Provider: OPENROUTER
Model: openai/gpt-oss-20b:free
API Key: ***
Embedding: sentence-transformers/all-MiniLM-L6-v2



#### из файлов

In [10]:
import pandas as pd
import sqlite3
# from typing import List, Dict, Any

class FlightSearch:
    def __init__(self, csv_path: str = 'flights.csv'):
        # Загружаем CSV 
        self.df = pd.read_csv(csv_path, sep=',')
        print("Колонки в загруженном CSV:", list(self.df.columns))

        # Создаём in-memory SQLite
        self.conn = sqlite3.connect(':memory:')
        self.df.to_sql('flights', self.conn, index=False, if_exists='replace')
        # Для удобства сохраняем курсор
        self.cursor = self.conn.cursor()
    
    def search_direct(self, origin: str, destination: str, date: str) -> pd.DataFrame:
        """
        Поиск прямых рейсов по точному совпадению городов и даты.
        origin, destination — названия городов (как в файле)
        date — строка в формате 'YYYY-MM-DD'
        """
        query = """
            SELECT line, flight, departure, arrival, departure_date, price
            FROM flights
            WHERE departure = ? AND arrival = ? AND departure_date = ?
            ORDER BY price
        """
        return pd.read_sql_query(query, self.conn, params=(origin, destination, date))
    
    def search_connecting(self, origin: str, destination: str, date: str, max_layover_days: int = 1) -> List[Dict[str, Any]]:
        """
        Поиск стыковочных рейсов (с одной пересадкой).
        Первый рейс вылетает точно в указанную дату.
        Второй рейс вылетает из города пересадки не ранее даты первого рейса
        и не позже date + max_layover_days.
        Возвращает список маршрутов с информацией о пересадке.
        """
        # Шаг 1: Найти все возможные города пересадки (куда можно улететь из origin в нужную дату)
        query_first = """
            SELECT DISTINCT arrival AS stopover
            FROM flights
            WHERE departure = ? AND departure_date = ?
        """
        stopovers = pd.read_sql_query(query_first, self.conn, params=(origin, date))['stopover'].tolist()
        
        if not stopovers:
            return []
        
        # Шаг 2: Для каждого города пересадки ищем рейсы до destination
        # Формируем условие для даты второго рейса: между date и date+max_layover_days
        date_end = pd.to_datetime(date) + pd.Timedelta(days=max_layover_days)
        date_end_str = date_end.strftime('%Y-%m-%d')
        
        results = []
        for stop in stopovers:
            query_second = """
                SELECT line, flight, departure, arrival, departure_date, price
                FROM flights
                WHERE departure = ? AND arrival = ? AND departure_date BETWEEN ? AND ?
                ORDER BY departure_date, price
            """
            second_flights = pd.read_sql_query(query_second, self.conn, params=(stop, destination, date, date_end_str))
            if second_flights.empty:
                continue
            
            # Получаем первый рейс (их может быть несколько)
            first_flights = pd.read_sql_query(
                "SELECT * FROM flights WHERE departure = ? AND arrival = ? AND departure_date = ?",
                self.conn, params=(origin, stop, date)
            )
            
            for _, first in first_flights.iterrows():
                for _, second in second_flights.iterrows():
                    # Проверяем, что второй рейс не раньше первого (по дате)
                    if second['departure_date'] < first['departure_date']:
                        continue
                    results.append({
                        'first_flight': {
                            'line': first['line'],
                            'flight': first['flight'],
                            'departure': first['departure'],
                            'arrival': first['arrival'],
                            'date': first['departure_date'],
                            'price': first['price']
                        },
                        'second_flight': {
                            'line': second['line'],
                            'flight': second['flight'],
                            'departure': second['departure'],
                            'arrival': second['arrival'],
                            'date': second['departure_date'],
                            'price': second['price']
                        },
                        'stopover_city': stop,
                        'total_price': first['price'] + second['price']
                    })
        
        # Сортируем по суммарной цене
        results.sort(key=lambda x: x['total_price'])
        return results
    
    def show_all_connecting(self) -> List[Dict[str, Any]]:
        """
        Все возможные стыковочные маршруты с одной пересадкой, где
        дата второго рейса не раньше даты первого и разница не более 2 дней.
        Возвращает список словарей с информацией о каждом стыковочном маршруте.
        """
        query = """
            SELECT 
                f1.line AS line1,
                f1.flight AS flight1,
                f1.departure AS from_city,
                f1.arrival AS stopover_city,
                f1.departure_date AS date1,
                f1.price AS price1,
                f2.line AS line2,
                f2.flight AS flight2,
                f2.departure AS stopover_city2,
                f2.arrival AS to_city,
                f2.departure_date AS date2,
                f2.price AS price2,
                (f1.price + f2.price) AS total_price
            FROM flights f1
            JOIN flights f2 ON f1.arrival = f2.departure
            WHERE 
                f2.departure_date >= f1.departure_date
                AND julianday(f2.departure_date) - julianday(f1.departure_date) <= 2
                AND f1.departure != f2.arrival   -- опционально: исключаем возврат в исходный город
            ORDER BY total_price
        """
        df = pd.read_sql_query(query, self.conn)
        
        results = []
        for _, row in df.iterrows():
            results.append({
                'first_flight': {
                    'line': row['line1'],
                    'flight': row['flight1'],
                    'departure': row['from_city'],
                    'arrival': row['stopover_city'],
                    'date': row['date1'],
                    'price': row['price1']
                },
                'second_flight': {
                    'line': row['line2'],
                    'flight': row['flight2'],
                    'departure': row['stopover_city2'],
                    'arrival': row['to_city'],
                    'date': row['date2'],
                    'price': row['price2']
                },
                'stopover_city': row['stopover_city'],
                'total_price': row['total_price']
            })
        return results

        
    
    def close(self):
        self.conn.close()

In [ ]:
# # Загрузка CSV
# flights_itab = pd.read_csv("data/flights.csv")
# print(f"✅ Загружено {len(flights_itab)} рейсов из data/flights.csv")
# print(flights_itab.head())
fs = FlightSearch('data/flights.csv')

# Прямые рейсы из Москвы в Санкт-Петербург 2026-01-14
direct = fs.search_direct('Москва', 'Санкт-Петербург', '2026-01-14')
print("Прямые рейсы:")
print(direct)

# Стыковочные рейсы
all_conn = fs.show_all_connecting()
print(f"✅ Найдено стыковочных маршрутов: {len(all_conn)}")

print("\nСтыковочные маршруты:")

for route in all_conn[:3]:  # покажем первые 3
    print(f"{route['first_flight']['flight']} {route['first_flight']['departure']}→{route['first_flight']['arrival']} ({route['first_flight']['date']}) + "
          f"{route['second_flight']['flight']} {route['second_flight']['departure']}→{route['second_flight']['arrival']} ({route['second_flight']['date']})  "
          f"Цена: {route['total_price']} руб.")

# Загрузка политики из data/policy.txt
with open("data/policy.txt", "r", encoding="utf-8") as f:
    policy_text = f.read()
print(f"✅ Загружен текст политики ({len(policy_text)} символов) из data/policy.txt")
print(policy_text[:500] + "..." if len(policy_text) > 500 else policy_text)

Колонки в загруженном CSV: ['line', 'flight', 'departure', 'arrival', 'departure_date', 'price']
Прямые рейсы:
     line  flight departure          arrival departure_date  price
0  Россия  РО5925    Москва  Санкт-Петербург     2026-01-14   6005
Найдено стыковочных маршрутов: 96

Стыковочные маршруты (первый рейс 2026-03-18):
РО5067 Москва→Иркутск (2026-12-08) + УР7353 Иркутск→Волгоград (2026-12-09)  Цена: 7854 руб.
ПО3046 Волгоград→Москва (2026-09-18) + ПО3624 Москва→Санкт-Петербург (2026-09-19)  Цена: 9489 руб.
НО6518 Волгоград→Иркутск (2026-03-15) + ПО3138 Иркутск→Воркута (2026-03-16)  Цена: 10054 руб.
✅ Загружен текст политики (5242 символов) из data/policy.txt
2. АВИАБИЛЕТЫ: МАКСИМАЛЬНАЯ ЦЕНА И КАТЕГОРИИ ПО ДОЛЖНОСТЯМ

2.1. Максимальная цена билета (туда-обратно, эконом-класс, включая сборы) не может превышать:
    - Для перелётов внутри РФ: 35 000 руб.
    - Для международных перелётов: 85 000 руб.

2.2. Категории бронирования в зависимости от должности сотрудника:

| Должность   

#### Embeddings

In [3]:
import shutil
import os

if os.path.exists('.data/.chroma'):
    shutil.rmtree('.data/.chroma')
    print("предыдущая бд удалена")

предыдущая бд удалена


In [ ]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


# Эмбеддер
try:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    print("✅ Эмбеддер загружен")
except Exception as e:
    print(f"⚠️ Ошибка загрузки эмбеддера: {e}")
    class DummyEmbeddings:
        def embed_documents(self, texts): return [[0.0]*384 for _ in texts]
        def embed_query(self, text): return [0.0]*384
    embeddings = DummyEmbeddings()
    print("⚠️ Используется DummyEmbeddings")

# Документы
policy_doc = Document(page_content=policy_text, metadata={"source": "policy"})

# ticket_docs = []
# for _, row in flights_itab.iterrows():
#     content = (
#         f"Рейс {row['flight']} "
#         f"{row['line']} "
#         f"{row['departure']} → {row['arrival']} "
#         f"{row['departure_date']}, "
#         f"{row['price']} руб., "
#         # f"{'прямой' if row['is_direct'] else 'с пересадкой'}"
#     )
#     metadata = {
#         "source": "flight",
#         "flight_number": row["flight"],
#         "departure_city": row["departure"],
#         "arrival_city": row["arrival"],
#         "price": row["price"],
#         "departure_date": row["departure_date"]
#     }
#     ticket_docs.append(Document(page_content=content, metadata=metadata))

# Чанкинг только для политики
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
policy_chunks = splitter.split_documents([policy_doc])
print(f"Политика разбита на {len(policy_chunks)} чанков")

# Векторное хранилище policy
# vectorstore = Chroma.from_documents(documents= policy_chunks + ticket_docs,
vectorstore = Chroma.from_documents(documents= policy_chunks,
                                    embedding=embeddings, # type: ignore
                                    persist_directory='.data/.chroma' )
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ policy vector store готов")

Прямые рейсы:
     line  flight departure          arrival departure_date  price
0  Россия  РО5925    Москва  Санкт-Петербург     2026-01-14   6005

Стыковочные маршруты (первый рейс 2026-03-18):


z:\temp\ipykernel_12736\2779434463.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2606.01it/s]


✅ Эмбеддер загружен
Политика разбита на 17 чанков
✅ policy vector store готов
